<a href="https://colab.research.google.com/github/cn8972/Echo-Bot/blob/main/Cognitive%20Offloading%20Dataset%20Explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import pandas as pd
import numpy as np
import gradio as gr
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report

DATA_SHEET = "Data_v2"

REQUIRED_COLUMNS = [
    "Occupation", "AI_Usage_Level", "Cognitive_Offloading_Level",
    "AI_Use_Frequency", "Verification_Rate", "Performance_Drop_Without_AI",
    "Cognitive_Offloading_Index"
]

MODEL_FEATURES = [
    "Occupation",
    "AI_Usage_Level",
    "AI_Use_Frequency",
    "Verification_Rate",
    "Performance_Drop_Without_AI",
    "industry_sector",
    "domain_expertise_level",
    "ai_literacy_level",
    "task_type",
    "task_complexity",
    "task_criticality",
    "time_pressure_level",
    "ai_tool_name",
    "ai_tool_type",
    "ai_prompt_count",
    "ai_output_acceptance_rate",
    "ai_output_edit_distance",
    "verification_method_primary",
    "verification_time_seconds",
    "task_accuracy_score",
    "error_count",
    "error_severity_max",
    "perceived_mental_workload",
    "cognitive_fatigue_level",
    "trust_in_ai_score",
    "perceived_autonomy_score",
    "organization_ai_policy_level",
    "workload_level",
    "interruptions_count",
    "offloading_type",
]

cat_features = [
    "Occupation",
    "AI_Usage_Level",
    "industry_sector",
    "domain_expertise_level",
    "ai_literacy_level",
    "task_type",
    "task_complexity",
    "task_criticality",
    "time_pressure_level",
    "ai_tool_name",
    "ai_tool_type",
    "verification_method_primary",
    "error_severity_max",
    "organization_ai_policy_level",
    "workload_level",
    "offloading_type",
]

num_features = [
    "AI_Use_Frequency",
    "Verification_Rate",
    "Performance_Drop_Without_AI",
    "ai_prompt_count",
    "ai_output_acceptance_rate",
    "ai_output_edit_distance",
    "verification_time_seconds",
    "task_accuracy_score",
    "error_count",
    "perceived_mental_workload",
    "cognitive_fatigue_level",
    "trust_in_ai_score",
    "perceived_autonomy_score",
    "interruptions_count",
]

APP_STATE = {
    "df": None,
    "reg_model": None,
    "clf_model": None,
    "REG_MAE": None,
    "REG_R2": None,
    "CLF_ACC": None,
    "CLF_REPORT": None,
    "X_train": None,
    "X_test": None,
    "occupation_choices": [],
    "ai_usage_choices": [],
    "sector_choices": [],
    "tool_choices": [],
    "task_type_choices": [],
    "offloading_choices": [],
    "domain_choices": [],
    "literacy_choices": [],
    "complexity_choices": [],
    "criticality_choices": [],
    "pressure_choices": [],
    "tool_type_choices": [],
    "verify_choices": [],
    "error_severity_choices": [],
    "policy_choices": [],
    "workload_choices": [],
    "offloading_type_choices": [],
}

NUMERIC_SUMMARY_COLUMNS = [
    "AI_Use_Frequency",
    "Verification_Rate",
    "Performance_Drop_Without_AI",
    "Cognitive_Offloading_Index",
    "ai_prompt_count",
    "ai_output_acceptance_rate",
    "ai_output_edit_distance",
    "verification_time_seconds",
    "task_accuracy_score",
    "error_count",
    "perceived_mental_workload",
    "cognitive_fatigue_level",
    "trust_in_ai_score",
    "perceived_autonomy_score",
    "interruptions_count",
]

COMMON_GROUP_COLUMNS = [
    "Occupation",
    "AI_Usage_Level",
    "industry_sector",
    "task_type",
    "task_complexity",
    "task_criticality",
    "ai_tool_name",
    "ai_tool_type",
    "verification_method_primary",
    "organization_ai_policy_level",
    "workload_level",
    "offloading_type",
    "Cognitive_Offloading_Level",
    "domain_expertise_level",
    "ai_literacy_level",
]

def load_data(file_obj, sheet_name=DATA_SHEET):
    if file_obj is None:
        raise ValueError("Please upload an Excel dataset file.")

    file_path = file_obj.name
    df = pd.read_excel(file_path, sheet_name=sheet_name)

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "timestamp_utc" in df.columns:
        df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce")

    return df

def train_models(df):
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
            ("num", "passthrough", num_features),
        ]
    )

    reg_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                min_samples_leaf=2,
                n_jobs=-1,
            )),
        ]
    )

    clf_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                min_samples_leaf=2,
                n_jobs=-1,
            )),
        ]
    )

    X = df[MODEL_FEATURES].copy()
    y_reg = df["Cognitive_Offloading_Index"].copy()
    y_clf = df["Cognitive_Offloading_Level"].copy()

    X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
        X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
    )

    reg_model.fit(X_train, y_reg_train)
    clf_model.fit(X_train, y_clf_train)

    reg_preds = reg_model.predict(X_test)
    clf_preds = clf_model.predict(X_test)

    REG_MAE = float(mean_absolute_error(y_reg_test, reg_preds))
    REG_R2 = float(r2_score(y_reg_test, reg_preds))
    CLF_ACC = float(accuracy_score(y_clf_test, clf_preds))
    CLF_REPORT = classification_report(y_clf_test, clf_preds, zero_division=0)

    return reg_model, clf_model, REG_MAE, REG_R2, CLF_ACC, CLF_REPORT, X_train, X_test

def model_summary_text():
    if APP_STATE["df"] is None:
        return "No model available. Please upload a dataset first."

    return (
        f"Training records: {len(APP_STATE['X_train']):,}\n"
        f"Test records: {len(APP_STATE['X_test']):,}\n"
        f"Regression MAE: {APP_STATE['REG_MAE']:.3f}\n"
        f"Regression R²: {APP_STATE['REG_R2']:.3f}\n"
        f"Classification Accuracy: {APP_STATE['CLF_ACC']:.3f}\n\n"
        "Classification Report:\n"
        f"{APP_STATE['CLF_REPORT']}"
    )

def get_feature_importances(top_n=20):
    require_data()
    model = APP_STATE["reg_model"].named_steps["model"]
    pre = APP_STATE["reg_model"].named_steps["preprocessor"]
    cat_names = pre.named_transformers_["cat"].get_feature_names_out(cat_features)
    all_features = list(cat_names) + num_features

    imp_df = pd.DataFrame({
        "feature": all_features,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    return imp_df.head(top_n)

def initialize_app(file_obj):
    try:
        df = load_data(file_obj)
        reg_model, clf_model, REG_MAE, REG_R2, CLF_ACC, CLF_REPORT, X_train, X_test = train_models(df)

        APP_STATE["df"] = df
        APP_STATE["reg_model"] = reg_model
        APP_STATE["clf_model"] = clf_model
        APP_STATE["REG_MAE"] = REG_MAE
        APP_STATE["REG_R2"] = REG_R2
        APP_STATE["CLF_ACC"] = CLF_ACC
        APP_STATE["CLF_REPORT"] = CLF_REPORT
        APP_STATE["X_train"] = X_train
        APP_STATE["X_test"] = X_test

        APP_STATE["occupation_choices"] = sorted(df["Occupation"].dropna().unique().tolist()) if "Occupation" in df.columns else []
        APP_STATE["ai_usage_choices"] = sorted(df["AI_Usage_Level"].dropna().unique().tolist()) if "AI_Usage_Level" in df.columns else []
        APP_STATE["sector_choices"] = sorted(df["industry_sector"].dropna().unique().tolist()) if "industry_sector" in df.columns else []
        APP_STATE["tool_choices"] = sorted(df["ai_tool_name"].dropna().unique().tolist()) if "ai_tool_name" in df.columns else []
        APP_STATE["task_type_choices"] = sorted(df["task_type"].dropna().unique().tolist()) if "task_type" in df.columns else []
        APP_STATE["offloading_choices"] = sorted(df["Cognitive_Offloading_Level"].dropna().unique().tolist()) if "Cognitive_Offloading_Level" in df.columns else []
        APP_STATE["domain_choices"] = sorted(df["domain_expertise_level"].dropna().unique().tolist()) if "domain_expertise_level" in df.columns else []
        APP_STATE["literacy_choices"] = sorted(df["ai_literacy_level"].dropna().unique().tolist()) if "ai_literacy_level" in df.columns else []
        APP_STATE["complexity_choices"] = sorted(df["task_complexity"].dropna().unique().tolist()) if "task_complexity" in df.columns else []
        APP_STATE["criticality_choices"] = sorted(df["task_criticality"].dropna().unique().tolist()) if "task_criticality" in df.columns else []
        APP_STATE["pressure_choices"] = sorted(df["time_pressure_level"].dropna().unique().tolist()) if "time_pressure_level" in df.columns else []
        APP_STATE["tool_type_choices"] = sorted(df["ai_tool_type"].dropna().unique().tolist()) if "ai_tool_type" in df.columns else []
        APP_STATE["verify_choices"] = sorted(df["verification_method_primary"].dropna().unique().tolist()) if "verification_method_primary" in df.columns else []
        APP_STATE["error_severity_choices"] = sorted(df["error_severity_max"].dropna().unique().tolist()) if "error_severity_max" in df.columns else []
        APP_STATE["policy_choices"] = sorted(df["organization_ai_policy_level"].dropna().unique().tolist()) if "organization_ai_policy_level" in df.columns else []
        APP_STATE["workload_choices"] = sorted(df["workload_level"].dropna().unique().tolist()) if "workload_level" in df.columns else []
        APP_STATE["offloading_type_choices"] = sorted(df["offloading_type"].dropna().unique().tolist()) if "offloading_type" in df.columns else []

        status = (
            f"Dataset loaded successfully.\n"
            f"Records: {len(df):,}\n"
            f"Columns: {len(df.columns):,}\n"
            f"Occupations: {df['Occupation'].nunique() if 'Occupation' in df.columns else 0}\n"
            f"Regression R²: {REG_R2:.3f}\n"
            f"Classification Accuracy: {CLF_ACC:.3f}"
        )

        default = lambda lst: lst[0] if lst else None

        return (
            status,
            gr.update(choices=APP_STATE["occupation_choices"]),
            gr.update(choices=APP_STATE["ai_usage_choices"]),
            gr.update(choices=APP_STATE["sector_choices"]),
            gr.update(choices=APP_STATE["tool_choices"]),
            gr.update(choices=APP_STATE["task_type_choices"]),
            gr.update(choices=APP_STATE["offloading_choices"]),
            gr.update(choices=APP_STATE["occupation_choices"], value=default(APP_STATE["occupation_choices"])),
            gr.update(choices=APP_STATE["ai_usage_choices"], value=default(APP_STATE["ai_usage_choices"])),
            gr.update(choices=APP_STATE["sector_choices"], value=default(APP_STATE["sector_choices"])),
            gr.update(choices=APP_STATE["domain_choices"], value=default(APP_STATE["domain_choices"])),
            gr.update(choices=APP_STATE["literacy_choices"], value=default(APP_STATE["literacy_choices"])),
            gr.update(choices=APP_STATE["task_type_choices"], value=default(APP_STATE["task_type_choices"])),
            gr.update(choices=APP_STATE["complexity_choices"], value=default(APP_STATE["complexity_choices"])),
            gr.update(choices=APP_STATE["criticality_choices"], value=default(APP_STATE["criticality_choices"])),
            gr.update(choices=APP_STATE["pressure_choices"], value=default(APP_STATE["pressure_choices"])),
            gr.update(choices=APP_STATE["tool_choices"], value=default(APP_STATE["tool_choices"])),
            gr.update(choices=APP_STATE["tool_type_choices"], value=default(APP_STATE["tool_type_choices"])),
            gr.update(choices=APP_STATE["verify_choices"], value=default(APP_STATE["verify_choices"])),
            gr.update(choices=APP_STATE["error_severity_choices"], value=default(APP_STATE["error_severity_choices"])),
            gr.update(choices=APP_STATE["policy_choices"], value=default(APP_STATE["policy_choices"])),
            gr.update(choices=APP_STATE["workload_choices"], value=default(APP_STATE["workload_choices"])),
            gr.update(choices=APP_STATE["offloading_type_choices"], value=default(APP_STATE["offloading_type_choices"])),
            model_summary_text(),
            get_feature_importances(20),
            plot_feature_importance(),
            []
        )

    except Exception as e:
        return (
            f"Error loading dataset: {str(e)}",
            gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update(),
            gr.update(), gr.update(), gr.update(), gr.update(), gr.update(),
            gr.update(), gr.update(), gr.update(), gr.update(), gr.update(),
            gr.update(), gr.update(), gr.update(), gr.update(), gr.update(),
            gr.update(), gr.update(),
            "No model available.",
            pd.DataFrame({"message": [str(e)]}),
            empty_plot(f"Error: {str(e)}"),
            []
        )

def require_data():
    if APP_STATE["df"] is None:
        raise ValueError("Please upload and load the dataset first.")

def filter_df(occupations, ai_usage_levels, sectors, tools, task_types, offloading_levels):
    require_data()
    filtered = APP_STATE["df"].copy()

    if occupations:
        filtered = filtered[filtered["Occupation"].isin(occupations)]
    if ai_usage_levels:
        filtered = filtered[filtered["AI_Usage_Level"].isin(ai_usage_levels)]
    if sectors:
        filtered = filtered[filtered["industry_sector"].isin(sectors)]
    if tools:
        filtered = filtered[filtered["ai_tool_name"].isin(tools)]
    if task_types:
        filtered = filtered[filtered["task_type"].isin(task_types)]
    if offloading_levels:
        filtered = filtered[filtered["Cognitive_Offloading_Level"].isin(offloading_levels)]

    return filtered

def overview_text(filtered):
    if filtered.empty:
        return "No records match the current filters."

    corr_cols = [c for c in [
        "AI_Use_Frequency",
        "Verification_Rate",
        "Performance_Drop_Without_AI",
        "ai_prompt_count",
        "task_accuracy_score",
        "perceived_mental_workload",
        "cognitive_fatigue_level",
        "trust_in_ai_score",
        "Cognitive_Offloading_Index",
    ] if c in filtered.columns]

    if "Cognitive_Offloading_Index" in corr_cols:
        corr = (
            filtered[corr_cols]
            .corr(numeric_only=True)["Cognitive_Offloading_Index"]
            .drop("Cognitive_Offloading_Index", errors="ignore")
            .sort_values(ascending=False)
        )
        top_pos = corr.head(3)
        top_neg = corr.tail(3)
        corr_text = (
            "Strongest positive correlations with Offloading Index:\n"
            + "\n".join([f"- {k}: {v:.3f}" for k, v in top_pos.items()])
            + "\n\nStrongest negative correlations with Offloading Index:\n"
            + "\n".join([f"- {k}: {v:.3f}" for k, v in top_neg.items()])
        )
    else:
        corr_text = "Correlation summary unavailable."

    return (
        f"Records: {len(filtered):,}\n"
        f"Unique occupations: {filtered['Occupation'].nunique() if 'Occupation' in filtered.columns else 0}\n"
        f"Unique sectors: {filtered['industry_sector'].nunique() if 'industry_sector' in filtered.columns else 0}\n"
        f"Mean Offloading Index: {filtered['Cognitive_Offloading_Index'].mean():.2f}\n"
        f"Median Offloading Index: {filtered['Cognitive_Offloading_Index'].median():.2f}\n"
        f"Mean AI Use Frequency: {filtered['AI_Use_Frequency'].mean():.3f}\n"
        f"Mean Verification Rate: {filtered['Verification_Rate'].mean():.3f}\n"
        f"Mean Task Accuracy: {filtered['task_accuracy_score'].mean():.2f}\n\n"
        f"{corr_text}"
    )

def descriptive_table(filtered):
    if filtered.empty:
        return pd.DataFrame({"message": ["No matching records"]})

    cols = [c for c in [
        "AI_Use_Frequency",
        "Verification_Rate",
        "Performance_Drop_Without_AI",
        "Cognitive_Offloading_Index",
        "ai_prompt_count",
        "ai_output_acceptance_rate",
        "verification_time_seconds",
        "task_accuracy_score",
        "perceived_mental_workload",
        "cognitive_fatigue_level",
        "trust_in_ai_score",
        "perceived_autonomy_score",
    ] if c in filtered.columns]
    return filtered[cols].describe().round(3).reset_index()

def sample_table(filtered):
    if filtered.empty:
        return pd.DataFrame({"message": ["No matching records"]})

    cols = [c for c in [
        "Record_ID", "Occupation", "industry_sector", "AI_Usage_Level",
        "Cognitive_Offloading_Level", "AI_Use_Frequency", "Verification_Rate",
        "Performance_Drop_Without_AI", "Cognitive_Offloading_Index",
        "ai_tool_name", "task_type", "task_complexity", "task_accuracy_score",
        "perceived_mental_workload", "cognitive_fatigue_level", "trust_in_ai_score",
    ] if c in filtered.columns]
    return filtered[cols].head(100)

def empty_plot(message="No data"):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.text(0.5, 0.5, message, ha="center", va="center")
    ax.axis("off")
    plt.tight_layout()
    return fig

def plot_distribution(filtered):
    if filtered.empty or "Cognitive_Offloading_Index" not in filtered.columns:
        return empty_plot("No matching records")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(filtered["Cognitive_Offloading_Index"].dropna(), bins=20)
    ax.set_title("Distribution of Cognitive Offloading Index")
    ax.set_xlabel("Cognitive Offloading Index")
    ax.set_ylabel("Count")
    plt.tight_layout()
    return fig

def plot_ai_usage_vs_offloading(filtered):
    if filtered.empty or "AI_Use_Frequency" not in filtered.columns or "Cognitive_Offloading_Index" not in filtered.columns:
        return empty_plot("No matching records")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(filtered["AI_Use_Frequency"], filtered["Cognitive_Offloading_Index"], alpha=0.6)
    ax.set_title("AI Use Frequency vs. Cognitive Offloading Index")
    ax.set_xlabel("AI Use Frequency")
    ax.set_ylabel("Cognitive Offloading Index")
    plt.tight_layout()
    return fig

def plot_group_means(filtered, group_col):
    if filtered.empty or group_col not in filtered.columns or "Cognitive_Offloading_Index" not in filtered.columns:
        return empty_plot("No matching records")
    fig, ax = plt.subplots(figsize=(10, 5))
    group_df = (
        filtered.groupby(group_col, observed=False)["Cognitive_Offloading_Index"]
        .mean()
        .sort_values(ascending=False)
        .head(15)
    )
    ax.barh(group_df.index[::-1], group_df.values[::-1])
    ax.set_title(f"Mean Cognitive Offloading Index by {group_col}")
    ax.set_xlabel("Mean Offloading Index")
    ax.set_ylabel(group_col)
    plt.tight_layout()
    return fig

def plot_feature_importance():
    if APP_STATE["df"] is None:
        return empty_plot("Please upload a dataset first")
    fig, ax = plt.subplots(figsize=(10, 7))
    imp_df = get_feature_importances(20)
    ax.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
    ax.set_title("Top Model Feature Importances")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    return fig

def predict_offloading(
    occupation,
    ai_usage_level,
    ai_use_frequency,
    verification_rate,
    performance_drop_without_ai,
    industry_sector,
    domain_expertise_level,
    ai_literacy_level,
    task_type,
    task_complexity,
    task_criticality,
    time_pressure_level,
    ai_tool_name,
    ai_tool_type,
    ai_prompt_count,
    ai_output_acceptance_rate,
    ai_output_edit_distance,
    verification_method_primary,
    verification_time_seconds,
    task_accuracy_score,
    error_count,
    error_severity_max,
    perceived_mental_workload,
    cognitive_fatigue_level,
    trust_in_ai_score,
    perceived_autonomy_score,
    organization_ai_policy_level,
    workload_level,
    interruptions_count,
    offloading_type,
):
    require_data()

    input_df = pd.DataFrame([{
        "Occupation": occupation,
        "AI_Usage_Level": ai_usage_level,
        "AI_Use_Frequency": float(ai_use_frequency),
        "Verification_Rate": float(verification_rate),
        "Performance_Drop_Without_AI": float(performance_drop_without_ai),
        "industry_sector": industry_sector,
        "domain_expertise_level": domain_expertise_level,
        "ai_literacy_level": ai_literacy_level,
        "task_type": task_type,
        "task_complexity": task_complexity,
        "task_criticality": task_criticality,
        "time_pressure_level": time_pressure_level,
        "ai_tool_name": ai_tool_name,
        "ai_tool_type": ai_tool_type,
        "ai_prompt_count": float(ai_prompt_count),
        "ai_output_acceptance_rate": float(ai_output_acceptance_rate),
        "ai_output_edit_distance": float(ai_output_edit_distance),
        "verification_method_primary": verification_method_primary,
        "verification_time_seconds": float(verification_time_seconds),
        "task_accuracy_score": float(task_accuracy_score),
        "error_count": float(error_count),
        "error_severity_max": error_severity_max,
        "perceived_mental_workload": float(perceived_mental_workload),
        "cognitive_fatigue_level": float(cognitive_fatigue_level),
        "trust_in_ai_score": float(trust_in_ai_score),
        "perceived_autonomy_score": float(perceived_autonomy_score),
        "organization_ai_policy_level": organization_ai_policy_level,
        "workload_level": workload_level,
        "interruptions_count": float(interruptions_count),
        "offloading_type": offloading_type,
    }])

    pred_index = float(APP_STATE["reg_model"].predict(input_df)[0])
    pred_level = str(APP_STATE["clf_model"].predict(input_df)[0])

    return (
        f"Predicted Cognitive Offloading Index: {pred_index:.2f}\n"
        f"Predicted Cognitive Offloading Level: {pred_level}"
    )

def update_dashboard(occupations, ai_usage_levels, sectors, tools, task_types, offloading_levels, group_col):
    try:
        filtered = filter_df(occupations, ai_usage_levels, sectors, tools, task_types, offloading_levels)
        return (
            overview_text(filtered),
            descriptive_table(filtered),
            sample_table(filtered),
            plot_distribution(filtered),
            plot_ai_usage_vs_offloading(filtered),
            plot_group_means(filtered, group_col),
        )
    except Exception as e:
        error_df = pd.DataFrame({"message": [str(e)]})
        return str(e), error_df, error_df, empty_plot(str(e)), empty_plot(str(e)), empty_plot(str(e))

# -----------------------------
# ChatGPT-like dataset interface
# -----------------------------

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

def safe_markdown_table(df, rows=12):
    if df is None or len(df) == 0:
        return "No rows to display."
    try:
        return df.head(rows).to_markdown(index=False)
    except Exception:
        return df.head(rows).to_string(index=False)

def find_best_column(user_text, df):
    text = normalize_text(user_text)
    candidates = list(df.columns)

    alias_map = {
        "occupation": "Occupation",
        "job": "Occupation",
        "sector": "industry_sector",
        "industry": "industry_sector",
        "ai usage": "AI_Usage_Level",
        "offloading level": "Cognitive_Offloading_Level",
        "offloading index": "Cognitive_Offloading_Index",
        "ai frequency": "AI_Use_Frequency",
        "verification rate": "Verification_Rate",
        "performance drop": "Performance_Drop_Without_AI",
        "tool": "ai_tool_name",
        "tool type": "ai_tool_type",
        "task": "task_type",
        "complexity": "task_complexity",
        "criticality": "task_criticality",
        "trust": "trust_in_ai_score",
        "fatigue": "cognitive_fatigue_level",
        "workload": "perceived_mental_workload",
        "accuracy": "task_accuracy_score",
        "autonomy": "perceived_autonomy_score",
        "prompt count": "ai_prompt_count",
        "interruptions": "interruptions_count",
        "policy": "organization_ai_policy_level",
        "verification time": "verification_time_seconds",
        "error count": "error_count",
        "offloading type": "offloading_type",
        "literacy": "ai_literacy_level",
        "domain expertise": "domain_expertise_level",
        "time pressure": "time_pressure_level",
        "acceptance rate": "ai_output_acceptance_rate",
        "edit distance": "ai_output_edit_distance",
    }

    for alias, col in alias_map.items():
        if alias in text and col in candidates:
            return col

    for col in candidates:
        if normalize_text(col).replace("_", " ") in text:
            return col

    return None

def apply_text_filters(df, user_text):
    text = normalize_text(user_text)
    filtered = df.copy()
    filters_used = []

    searchable_cols = [
        "Occupation",
        "AI_Usage_Level",
        "industry_sector",
        "ai_tool_name",
        "task_type",
        "Cognitive_Offloading_Level",
        "domain_expertise_level",
        "ai_literacy_level",
        "task_complexity",
        "task_criticality",
        "time_pressure_level",
        "ai_tool_type",
        "verification_method_primary",
        "error_severity_max",
        "organization_ai_policy_level",
        "workload_level",
        "offloading_type",
    ]

    for col in searchable_cols:
        if col in filtered.columns:
            values = filtered[col].dropna().astype(str).unique().tolist()
            for v in values:
                if normalize_text(v) in text:
                    filtered = filtered[filtered[col].astype(str) == v]
                    filters_used.append(f"{col} = {v}")

    return filtered, filters_used

def numeric_column_from_text(user_text, df):
    target_col = find_best_column(user_text, df)
    if target_col and pd.api.types.is_numeric_dtype(df[target_col]):
        return target_col

    for col in NUMERIC_SUMMARY_COLUMNS:
        if col in df.columns and normalize_text(col).replace("_", " ") in normalize_text(user_text):
            return col

    if "Cognitive_Offloading_Index" in df.columns:
        return "Cognitive_Offloading_Index"
    return None

def grouped_mean_response(df, metric_col, group_col, top_n=15):
    grouped = (
        df.groupby(group_col, observed=False)[metric_col]
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
        .reset_index()
    )
    grouped.columns = [group_col, f"mean_{metric_col}"]
    return grouped

def correlation_response(df, target_col):
    numeric_df = df.select_dtypes(include=[np.number]).copy()
    if target_col not in numeric_df.columns:
        return None
    corr = numeric_df.corr(numeric_only=True)[target_col].drop(target_col).sort_values(ascending=False)
    out = pd.DataFrame({
        "variable": corr.index,
        "correlation_with_target": corr.values
    })
    return out

def answer_dataset_question(message, history):
    try:
        require_data()
        df = APP_STATE["df"].copy()
        user_text = normalize_text(message)

        filtered, filters_used = apply_text_filters(df, user_text)
        filter_note = ""
        if filters_used:
            filter_note = "Active inferred filters: " + "; ".join(filters_used) + "\n\n"

        if "help" in user_text or "what can you do" in user_text:
            return (
                "You can ask things such as:\n"
                "- Show descriptive statistics\n"
                "- How many records are in healthcare?\n"
                "- Average cognitive offloading index by occupation\n"
                "- Correlation with cognitive offloading index\n"
                "- Top 10 records by task accuracy\n"
                "- Show records for finance and high AI usage\n"
                "- Compare trust in AI by industry sector\n"
                "- What is the mean verification rate?\n"
            )

        if "columns" in user_text or "variables" in user_text:
            cols = pd.DataFrame({"columns": df.columns})
            return filter_note + "Available columns:\n\n" + safe_markdown_table(cols, rows=len(cols))

        if "how many" in user_text or "count" in user_text or "number of records" in user_text:
            return filter_note + f"Matching records: **{len(filtered):,}**"

        if "descriptive" in user_text or "summary statistics" in user_text or "describe" in user_text:
            desc = filtered.select_dtypes(include=[np.number]).describe().round(3).reset_index()
            return filter_note + "Descriptive statistics:\n\n" + safe_markdown_table(desc, rows=20)

        if "top" in user_text and "by" in user_text:
            match_n = re.search(r"top\s+(\d+)", user_text)
            n = int(match_n.group(1)) if match_n else 10
            by_col = find_best_column(user_text.split("by", 1)[1], filtered) if "by" in user_text else None
            if by_col is None:
                return "I could not identify the ranking column. Try: 'Top 10 records by task accuracy score'."
            if by_col not in filtered.columns:
                return f"The column '{by_col}' is not available."
            ranked = filtered.sort_values(by=by_col, ascending=False).head(n)
            display_cols = [c for c in ["Record_ID", "Occupation", "industry_sector", by_col, "Cognitive_Offloading_Index"] if c in ranked.columns]
            return filter_note + f"Top {n} records by **{by_col}**:\n\n" + safe_markdown_table(ranked[display_cols], rows=n)

        if ("average" in user_text or "mean" in user_text) and "by" in user_text:
            left, right = user_text.split("by", 1)
            metric_col = numeric_column_from_text(left, filtered)
            group_col = find_best_column(right, filtered)
            if metric_col is None or group_col is None:
                return "I could not identify the metric and grouping columns. Try: 'Average cognitive offloading index by occupation'."
            grouped = grouped_mean_response(filtered, metric_col, group_col)
            return (
                filter_note
                + f"Average **{metric_col}** by **{group_col}**:\n\n"
                + safe_markdown_table(grouped, rows=20)
            )

        if ("median" in user_text) and "by" in user_text:
            left, right = user_text.split("by", 1)
            metric_col = numeric_column_from_text(left, filtered)
            group_col = find_best_column(right, filtered)
            if metric_col is None or group_col is None:
                return "I could not identify the metric and grouping columns."
            grouped = (
                filtered.groupby(group_col, observed=False)[metric_col]
                .median()
                .sort_values(ascending=False)
                .reset_index()
            )
            grouped.columns = [group_col, f"median_{metric_col}"]
            return (
                filter_note
                + f"Median **{metric_col}** by **{group_col}**:\n\n"
                + safe_markdown_table(grouped, rows=20)
            )

        if "correlation" in user_text or "correlate" in user_text:
            target_col = numeric_column_from_text(user_text, filtered)
            if target_col is None:
                return "I could not determine the target numeric variable for correlation analysis."
            corr_df = correlation_response(filtered, target_col)
            if corr_df is None:
                return f"No numeric correlation output available for {target_col}."
            return (
                filter_note
                + f"Correlations with **{target_col}**:\n\n"
                + safe_markdown_table(corr_df.head(15), rows=15)
            )

        if "show records" in user_text or "show rows" in user_text or "sample" in user_text:
            display_cols = [c for c in [
                "Record_ID", "Occupation", "industry_sector", "AI_Usage_Level",
                "Cognitive_Offloading_Level", "Cognitive_Offloading_Index",
                "AI_Use_Frequency", "Verification_Rate", "task_accuracy_score"
            ] if c in filtered.columns]
            return filter_note + "Sample records:\n\n" + safe_markdown_table(filtered[display_cols].head(15), rows=15)

        if "mean" in user_text or "average" in user_text:
            metric_col = numeric_column_from_text(user_text, filtered)
            if metric_col is None:
                return "I could not identify the numeric column."
            return filter_note + f"Mean **{metric_col}**: **{filtered[metric_col].mean():.3f}**"

        if "median" in user_text:
            metric_col = numeric_column_from_text(user_text, filtered)
            if metric_col is None:
                return "I could not identify the numeric column."
            return filter_note + f"Median **{metric_col}**: **{filtered[metric_col].median():.3f}**"

        if "unique" in user_text or "distinct" in user_text:
            col = find_best_column(user_text, filtered)
            if col is None:
                return "I could not identify the column."
            values = pd.DataFrame({col: sorted(filtered[col].dropna().astype(str).unique().tolist())})
            return filter_note + f"Distinct values for **{col}**:\n\n" + safe_markdown_table(values, rows=100)

        if "overview" in user_text or "summary" in user_text:
            return filter_note + overview_text(filtered)

        return (
            filter_note
            + "I could not fully interpret that request. Try one of these:\n"
            + "- Average cognitive offloading index by occupation\n"
            + "- Correlation with cognitive offloading index\n"
            + "- Show records for healthcare\n"
            + "- Top 10 records by task accuracy\n"
            + "- Descriptive statistics\n"
        )

    except Exception as e:
        return f"Error: {str(e)}"

def chat_submit(message, history):
    if history is None:
        history = []
    response = answer_dataset_question(message, history)
    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]
    return history, ""

group_options = [
    "Occupation",
    "AI_Usage_Level",
    "industry_sector",
    "task_type",
    "task_complexity",
    "task_criticality",
    "ai_tool_name",
    "ai_tool_type",
    "verification_method_primary",
    "organization_ai_policy_level",
    "workload_level",
    "offloading_type",
]

with gr.Blocks(title="Cognitive Offloading Dataset Explorer") as demo:
    gr.Markdown("# Cognitive Offloading Dataset Explorer")
    gr.Markdown(
        "Upload an Excel dataset first, then use the dashboard for filtering, descriptive analysis, visualization, predictive modeling, and conversational dataset exploration."
    )

    with gr.Tab("Upload Dataset"):
        file_input = gr.File(label="Upload Excel Dataset", file_types=[".xlsx"])
        load_btn = gr.Button("Load Dataset")
        load_status = gr.Textbox(label="Load Status", lines=6)

    with gr.Tab("Dataset Explorer"):
        with gr.Row():
            occupations = gr.Dropdown([], multiselect=True, label="Occupation")
            ai_usage_levels = gr.Dropdown([], multiselect=True, label="AI Usage Level")
            sectors = gr.Dropdown([], multiselect=True, label="Industry Sector")
        with gr.Row():
            tools = gr.Dropdown([], multiselect=True, label="AI Tool")
            task_types = gr.Dropdown([], multiselect=True, label="Task Type")
            offloading_levels = gr.Dropdown([], multiselect=True, label="Offloading Level")

        group_col = gr.Dropdown(group_options, value="Occupation", label="Group Comparison By")
        refresh_btn = gr.Button("Refresh Analysis")

        overview_box = gr.Textbox(label="Analytic Summary", lines=16)
        desc_table = gr.Dataframe(label="Descriptive Statistics")
        records_table = gr.Dataframe(label="Filtered Records")
        dist_plot = gr.Plot(label="Distribution Plot")
        scatter_plot = gr.Plot(label="Relationship Plot")
        group_plot = gr.Plot(label="Group Comparison Plot")

    with gr.Tab("Model Diagnostics"):
        model_box = gr.Textbox(label="Model Performance", lines=18, value="No model available. Please upload a dataset first.")
        feature_plot = gr.Plot(label="Feature Importance")
        feature_table = gr.Dataframe(label="Top Features")

    with gr.Tab("Predictive Scenario Tool"):
        with gr.Row():
            p_occupation = gr.Dropdown([], label="Occupation")
            p_ai_usage = gr.Dropdown([], label="AI Usage Level")
            p_sector = gr.Dropdown([], label="Industry Sector")
            p_domain = gr.Dropdown([], label="Domain Expertise")
            p_literacy = gr.Dropdown([], label="AI Literacy")

        with gr.Row():
            p_task_type = gr.Dropdown([], label="Task Type")
            p_task_complexity = gr.Dropdown([], label="Task Complexity")
            p_task_criticality = gr.Dropdown([], label="Task Criticality")
            p_time_pressure = gr.Dropdown([], label="Time Pressure")

        with gr.Row():
            p_tool_name = gr.Dropdown([], label="AI Tool Name")
            p_tool_type = gr.Dropdown([], label="AI Tool Type")
            p_verify_method = gr.Dropdown([], label="Verification Method")
            p_error_severity = gr.Dropdown([], label="Error Severity")

        with gr.Row():
            p_policy = gr.Dropdown([], label="Policy Level")
            p_workload = gr.Dropdown([], label="Workload Level")
            p_offloading_type = gr.Dropdown([], label="Offloading Type")

        p_ai_use_freq = gr.Slider(0, 1, value=0.5, step=0.01, label="AI Use Frequency")
        p_ver_rate = gr.Slider(0, 1, value=0.5, step=0.01, label="Verification Rate")
        p_perf_drop = gr.Slider(0, 1, value=0.3, step=0.01, label="Performance Drop Without AI")

        p_prompt_count = gr.Number(value=5, label="AI Prompt Count")
        p_accept_rate = gr.Slider(0, 1, value=0.5, step=0.01, label="AI Output Acceptance Rate")
        p_edit_distance = gr.Slider(0, 1, value=0.5, step=0.01, label="AI Output Edit Distance")

        p_ver_time = gr.Number(value=30, label="Verification Time Seconds")
        p_accuracy = gr.Number(value=75, label="Task Accuracy Score")
        p_error_count = gr.Number(value=1, label="Error Count")

        p_workload_score = gr.Number(value=50, label="Mental Workload")
        p_fatigue = gr.Number(value=50, label="Cognitive Fatigue")
        p_trust = gr.Number(value=50, label="Trust in AI")
        p_autonomy = gr.Number(value=50, label="Perceived Autonomy")
        p_interruptions = gr.Number(value=2, label="Interruptions Count")

        predict_btn = gr.Button("Run Scenario Prediction")
        prediction_output = gr.Textbox(label="Prediction Output", lines=5)

    with gr.Tab("Dataset Chat"):
        gr.Markdown(
            "Ask questions about the uploaded dataset in natural language. Examples: "
            "`show descriptive statistics`, `average cognitive offloading index by occupation`, "
            "`correlation with trust in ai score`, `show records for healthcare`, "
            "`top 10 records by task accuracy`."
        )
        chatbot = gr.Chatbot(label="Dataset Analyst", type="messages", height=500)
        chat_input = gr.Textbox(label="Ask a question about the dataset", placeholder="Example: average cognitive offloading index by industry sector")
        send_btn = gr.Button("Send")
        clear_btn = gr.Button("Clear Chat")

    load_btn.click(
        fn=initialize_app,
        inputs=[file_input],
        outputs=[
            load_status,
            occupations,
            ai_usage_levels,
            sectors,
            tools,
            task_types,
            offloading_levels,
            p_occupation,
            p_ai_usage,
            p_sector,
            p_domain,
            p_literacy,
            p_task_type,
            p_task_complexity,
            p_task_criticality,
            p_time_pressure,
            p_tool_name,
            p_tool_type,
            p_verify_method,
            p_error_severity,
            p_policy,
            p_workload,
            p_offloading_type,
            model_box,
            feature_table,
            feature_plot,
            chatbot,
        ],
    )

    refresh_btn.click(
        fn=update_dashboard,
        inputs=[occupations, ai_usage_levels, sectors, tools, task_types, offloading_levels, group_col],
        outputs=[overview_box, desc_table, records_table, dist_plot, scatter_plot, group_plot],
    )

    predict_btn.click(
        fn=predict_offloading,
        inputs=[
            p_occupation, p_ai_usage, p_ai_use_freq, p_ver_rate, p_perf_drop,
            p_sector, p_domain, p_literacy, p_task_type, p_task_complexity,
            p_task_criticality, p_time_pressure, p_tool_name, p_tool_type,
            p_prompt_count, p_accept_rate, p_edit_distance, p_verify_method,
            p_ver_time, p_accuracy, p_error_count, p_error_severity,
            p_workload_score, p_fatigue, p_trust, p_autonomy, p_policy,
            p_workload, p_interruptions, p_offloading_type,
        ],
        outputs=prediction_output,
    )

    send_btn.click(
        fn=chat_submit,
        inputs=[chat_input, chatbot],
        outputs=[chatbot, chat_input],
    )

    chat_input.submit(
        fn=chat_submit,
        inputs=[chat_input, chatbot],
        outputs=[chatbot, chat_input],
    )

    clear_btn.click(
        fn=lambda: [],
        inputs=[],
        outputs=[chatbot],
    )

if __name__ == "__main__":
    demo.launch()


/tmp/ipykernel_16015/2341947241.py:970: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Dataset Analyst", type="messages", height=500)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2b39a0acefb4f2f601.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
